# Multiagent Pipeline

Pensiamo se fare un indice
1. [Configuration](#configuration)

In [764]:
# Import Libraries
import io
import json
import math
import os
import re
import sys
import traceback
import unicodedata
from typing import Any, Callable, Iterable, Optional
import gradio as gr
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [765]:
# Load Dataset
df = pd.read_csv('data/raw/ALLARMI.csv')
df.head(5)

,OCCORRENZE,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,CITTA_PARTENZA,...,ZONA,TOT,MOTIVO_ALLARME,note_operatore,flag_rischio,Paese Partenza,CODICE PAESE ARR,3zona,paese%arr,tot voli
0,Voli con Allarmi,FCO,IST,2024,01,2024-01-30 09:15:00,Fiumicino,Havalimani,Roma,Istanbul,...,5,1,Manuale,NaN,NaN,Turchia,ITA,5,Italia,1
1,Viaggiatori con Allarmi,CIA,STN,2024,02,2024-02-03 13:15:00,Ciampino,Stansted,Roma,Londra,...,5,5,Manuale,NaN,NaN,Regno Unito,ITA,5,Italia,5
2,Viaggiatori entrati nel Sistema,FCO,LHR,2024,01,2024-01-15 08:45:00,Fiumicino,London Heathrow,Roma,Londra,...,5,110,TSC,NaN,NaN,Regno Unito,ITA,5,Italia,110
3,Voli con Allarmi,MXP,LHR,2024,02,2024-02-02 08:40:00,Malpensa,London Heathrow,Milano,Londra,...,2,1,SDI,NaN,NaN,Regno Unito,ITA,2,Italia,1
4,Viaggiatori con Allarmi,PSA,BRS,2024,02,2024-02-16 12:50:00,Galileo Galilei,Bristol,Pisa,Bristol,...,8,2,INTERPOL,NaN,NaN,Regno Unito,ITA,8,Italia,2


# Configuration

Here we set up the API connection, define all file paths (input, output, and intermediate files), and initialize basic parameters used across the pipeline.

In [766]:
load_dotenv()

MISTRAL_BASE_URL = "https://api.mistral.ai/v1"
MISTRAL_API_KEY  = os.getenv("MISTRAL_API_KEY", "")
MISTRAL_MODEL    = "mistral-small-latest"

_client = OpenAI(base_url=MISTRAL_BASE_URL, api_key=MISTRAL_API_KEY)

# Paths 
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
ALLARMI_CSV = os.path.join(RAW_DIR, "ALLARMI.csv")
TIPOLOGIA_CSV = os.path.join(RAW_DIR, "TIPOLOGIA_VIAGGIATORE.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
FINDINGS_JSON = os.path.join(OUTPUT_DIR, "findings.json")
COLUMN_PROFILES_JSON = os.path.join(OUTPUT_DIR, "column_profiles.json")
CLEANING_PLAN_JSON   = os.path.join(OUTPUT_DIR, "cleaning_plan.json")   # NEW: slim JSON for cleaning agent

# Profiling 
DOMINANT_FORMAT_SAMPLE_SIZE = 200 

## Cleaning Helpers

This section defines a set of reusable functions used for dataset cleaning and normalization.

- **`_strip_accents(text)`**  
  Removes accents and diacritics from text (e.g. "à" → "a").

- **`_collapse_spaces(text)`**  
  Normalizes whitespace by collapsing multiple spaces into one.

- **`to_snake_case(name)`**  
  Converts column names to a consistent `snake_case` format.  
  Handles special characters, camelCase, and noisy inputs.

- **`normalize_column_names(df)`**  
  Applies `snake_case` normalization to all column names in a DataFrame.

- **`deduplicate_column_names(df)`**  
  Ensures all column names are unique by adding suffixes (`__1`, `__2`, ...).

- **`remove_duplicate_rows(df)`**  
  Removes exact duplicate rows from the dataset.

- **`standardize_missing_values(df)`**  
  Converts common missing value placeholders (e.g. "n/a", "unknown") into `NaN`.

- **`normalize_text_values(df)`**  
  Cleans textual data by lowercasing and normalizing spaces.

- **`load_cleaning_plan(path, dataset_key)`**  
  Loads a JSON-based cleaning plan for a specific dataset, used to guide transformations.

These helpers provide a generic and reusable foundation for the data cleaning pipeline.

In [767]:
DEFAULT_CLEANING_MISSING_TOKENS = {
    "", " ", "na", "n/a", "null", "none", "nan", "unknown", "undefined",
    "-", "--", "not available", "missing", "n.d.", "nd"
}


def _strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def _collapse_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def to_snake_case(name: str) -> str:
    name = "" if name is None else str(name)
    name = _strip_accents(name).strip()

    # Separate camelCase / PascalCase boundaries
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", name)

    # Replace non-alphanumeric runs with underscore
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)

    # Collapse underscores and trim
    name = re.sub(r"_+", "_", name).strip("_").lower()

    if not name:
        name = "unnamed_column"

    # If the name starts with a digit, prefix it to keep it identifier-like
    if re.match(r"^\d", name):
        name = f"col_{name}"

    return name


def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [to_snake_case(col) for col in out.columns]
    return out


def deduplicate_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    seen = {}
    new_cols = []

    for col in out.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}__{seen[col]}")

    out.columns = new_cols
    return out


def remove_duplicate_rows(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates().copy()


def standardize_missing_values(
    df: pd.DataFrame,
    missing_tokens: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    out = df.copy()
    tokens = {str(x).strip().lower() for x in (missing_tokens or DEFAULT_CLEANING_MISSING_TOKENS)}

    for col in out.columns:
        out[col] = out[col].apply(
            lambda x: pd.NA
            if isinstance(x, str) and x.strip().lower() in tokens
            else x
        )
    return out


def normalize_text_values(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def _norm(x):
        if pd.isna(x):
            return x
        if isinstance(x, str):
            return _collapse_spaces(x).lower()
        return x

    for col in out.columns:
        out[col] = out[col].apply(_norm)

    return out


def load_cleaning_plan(plan_json_path: str, dataset_key: str) -> dict:
    with open(plan_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get(dataset_key, {})

## Column Roles & Data Types

This section defines a controlled vocabulary used to classify dataset columns during profiling.

- **Column Roles (`COLUMN_ROLES`)**  
  A predefined set of semantic roles (e.g. `count`, `measure`, `date`, `category`) that the LLM assigns to each column.  
  If no role fits, the model can propose a custom role (`custom:<name>`).

- **Role → Data Type Mapping (`ROLE_TO_EXPECTED_DTYPE`)**  
  Each role is mapped to a specific pandas dtype.  
  This ensures that, after cleaning, all columns are cast to a consistent and appropriate type.

- **`resolve_expected_dtype(role)`**  
  Given a role, returns the corresponding pandas dtype.  
  If the role is custom or unknown, it defaults to `"string"` for safety.

This mechanism allows the cleaning pipeline to remain **data-driven, consistent, and extensible** across different datasets.

In [768]:
COLUMN_ROLES = {

    # numeric
    "identifier_numeric",   # numeric IDs without arithmetic meaning (preserve as string to keep leading zeros)
    "count",                # non-negative integer counts (e.g. number of flights)
    "measure",              # continuous measurements (weight, distance, price)
    "percentage",           # percentage value
    "year",                 # year (e.g. 2024)
    "month_number",         # month as 01-12
    "day_number",           # day as 01-31

    # string / code
    "identifier",           # generic identifier (could be alphanumeric)
    "category",             # categorical variable (e.g. "low", "medium", "high")
    "free_text",            # free text / operator notes
    "flag_binary",          # binary flag (yes/no, 0/1, alto/basso)

    # temporal
    "date",                 # date without time
    "datetime",             # date + time

    # special
    "unknown",              # LLM cannot classify
}


ROLE_TO_EXPECTED_DTYPE = {
    "identifier_numeric": "string",      # keep as string to preserve leading zeros
    "count":              "Int64",       # nullable integer
    "measure":            "Float64",     # nullable float
    "percentage":         "Float64",
    "year":               "Int64",
    "month_number":       "Int64",
    "day_number":         "Int64",

    "identifier":         "string",
    "category":           "string",
    "free_text":          "string",
    "flag_binary":        "string",

    "date":               "datetime64[ns]",
    "datetime":           "datetime64[ns]",

    "unknown":            "string",      # safe default
}


def resolve_expected_dtype(role: str) -> str:
    if role in ROLE_TO_EXPECTED_DTYPE:
        return ROLE_TO_EXPECTED_DTYPE[role]
    if isinstance(role, str) and role.startswith("custom:"):
        return "string"
    return "string"

## JSON Helpers

This section provides utility functions to safely handle JSON serialization and file operations.

- **`_to_native(value)`**  
  Converts pandas and numpy objects into JSON-compatible Python types.  
  Ensures values like timestamps, arrays, and NaN are properly serialized.

- **`_safe_load_json(path)`**  
  Loads a JSON file safely.  
  Returns an empty dictionary if the file is missing, empty, or corrupted.

- **`_safe_write_json(path, data)`**  
  Writes data to a JSON file in a consistent format.  
  Automatically creates directories if needed and ensures proper encoding and formatting.

These helpers ensure robust and error-tolerant JSON handling across the pipeline.

In [769]:
def _to_native(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _to_native(v) for k, v in value.items()}
    if isinstance(value, (np.ndarray, list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if math.isnan(float(value)):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


def _safe_load_json(path: str) -> dict:
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, dict) else {}
    except Exception:
        return {}


def _safe_write_json(path: str, data: dict) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_native(data), f, ensure_ascii=False, indent=2)

## DataFrame Helpers

Utility functions for extracting structural information from pandas DataFrames,
such as shape, data types, and missing value statistics.

In [770]:
def get_dataframe_shape(df: pd.DataFrame) -> dict:
    rows, cols = df.shape
    return {"rows": int(rows), "columns": int(cols)}


def get_dataframe_dtypes(df: pd.DataFrame) -> dict:
    return {col: str(dtype) for col, dtype in df.dtypes.items()}


def get_column_missing_stats(df: pd.DataFrame) -> dict:
    result = {}
    for col in df.columns:
        mask = df[col].isna()
        result[col] = {
            "null_count": int(mask.sum()),
            "non_null_count": int((~mask).sum()),
            "missing_percentage": round(float(mask.mean() * 100), 2),
        }
    return result

## Format Detection

This section provides utilities to analyze the **structural format of column values**.

- **`_value_format_label(value)`**  
  Classifies a single value into a coarse format category  
  (e.g. `digit`, `float`, `alpha`, `alphanumeric`, `generic`, `missing`).

- **`detect_dominant_format(series)`**  
  Identifies the most common value format in a column based on a sample.  
  Returns the dominant label and its frequency.

- **`collect_non_conforming_samples(series, dominant_label)`**  
  Finds values that do not match the dominant format.  
  Useful to detect inconsistencies or dirty data.

- **`_try_numeric(series)`**  
  Attempts to convert a column to numeric format, handling common cases  
  like comma decimals or mixed string inputs.

### Design principle
These functions focus only on **syntactic patterns**, not semantics.  
Higher-level interpretation (e.g. understanding if a column is a date or ID)  
is delegated to the LLM during profiling.

In [771]:
def _value_format_label(value: Any) -> str:
    if value is None:
        return "missing"

    s = str(value).strip()

    if s == "":
        return "empty"

    if s.isdigit():
        return "digit"

    try:
        float(s.replace(",", "."))
        return "float"
    except Exception:
        pass

    if s.isalpha():
        return "alpha"

    has_alpha = any(c.isalpha() for c in s)
    has_digit = any(c.isdigit() for c in s)
    if has_alpha and has_digit:
        return "alphanumeric"

    return "generic"


def detect_dominant_format(
    series: pd.Series,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:

    head_sample = series.head(sample_size)
    clean = head_sample.dropna()

    if clean.empty:
        return {
            "label": None,
            "share_pct": 0.0,
            "sample_size": 0,
        }

    labels = clean.map(_value_format_label)
    counts = labels.value_counts(dropna=False)
    total = int(counts.sum())

    return {
        "label": str(counts.index[0]),
        "share_pct": round(float(counts.iloc[0] / total * 100), 2),
        "sample_size": total,
    }


def collect_non_conforming_samples(
    series: pd.Series,
    dominant_label: str,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    max_samples: int = 100,
) -> dict:

    head_sample = series.head(sample_size)
    clean = head_sample.dropna()

    if clean.empty or dominant_label is None:
        return {"share_pct": 0.0, "samples": []}

    labels = clean.map(_value_format_label)
    non_conforming_mask = labels != dominant_label
    non_conforming_values = clean.loc[non_conforming_mask]

    total = int(len(clean))
    n_non_conforming = int(non_conforming_mask.sum())

    unique_samples = []
    for v in non_conforming_values.tolist():
        native_v = _to_native(v)
        if native_v not in unique_samples:
            unique_samples.append(native_v)
        if len(unique_samples) >= max_samples:
            break

    return {
        "share_pct": round(float(n_non_conforming / total * 100), 2) if total else 0.0,
        "samples": unique_samples,  
    }


def _try_numeric(series: pd.Series) -> pd.Series:

    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    s = series.astype("string").str.strip()
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

## LLM-Driven Column Enrichment

This section uses the LLM to enrich each column profile with semantic information and transformation logic.

- **Goal**  
  Extend the deterministic profile of a column with:
  - a semantic **role** (e.g. `count`, `date`, `category`)
  - a **transformation rule** to clean non-conforming values
  - a short **description** for human interpretation

- **Transformation Actions (`TRANSFORMATION_ACTIONS`)**  
  The LLM must choose from a closed set of actions:
  - `none` → no changes needed  
  - `extract` → extract valid patterns (via regex)  
  - `map` → map values to canonical ones  
  - `set_null` → discard invalid values  

- **`_build_enrichment_prompt(profile)`**  
  Builds a structured prompt combining:
  - column statistics
  - dominant format
  - sample values  
  This ensures the LLM makes decisions based on actual data.

- **`_parse_llm_enrichment_response(response)`**  
  Parses the LLM output, handling formatting issues like markdown fences.

- **`_fallback_enrichment(profile, error)`**  
  Provides a safe default when the LLM fails or returns invalid output.

- **`_validate_and_fix_enrichment(enrichment, profile)`**  
  Applies deterministic checks to correct inconsistent LLM outputs  
  (e.g. numeric role assigned to textual data).

- **`enrich_column_profile_with_llm(profile, llm_callable)`**  
  Main function:
  - calls the LLM
  - parses and validates the response
  - returns a consistent, structured enrichment

### Design principle
The LLM handles **semantic interpretation**, while deterministic logic ensures  
**robustness, consistency, and safety** of the final output.

In [772]:
TRANSFORMATION_ACTIONS = {
    "none",        # all values already conform
    "extract",     # extract substring matching a pattern (params: regex)
    "map",         # map non-conforming values to canonical ones (params: mapping, canonical_values)
    "set_null",    # explicitly null out broken placeholders
}


def _build_enrichment_prompt(deterministic_profile: dict) -> str:
    return f"""
You are a data profiling assistant.

Given the deterministic profile of ONE column, enrich it with:
1. a ROLE from the closed vocabulary
2. a declarative TRANSFORMATION_RULE
3. a brief human-readable DESCRIPTION

ROLES:
{sorted(COLUMN_ROLES)}

If none fits, return "custom:<short_snake_case_name>".

ROLE TO EXPECTED DTYPE:
{json.dumps(ROLE_TO_EXPECTED_DTYPE, indent=2)}

TRANSFORMATION ACTIONS:
{sorted(TRANSFORMATION_ACTIONS)}

A transformation_rule MUST have this structure:
{{
  "action": "<one of the actions above>",
  "target_format": "<expected value format>",
  "params": {{...}},
  "description": "<what the rule does>"
}}

Params examples:
- extract: {{"regex": "\\\\d+"}}
- map: {{"mapping": {{"<raw_value>": "<canonical_value>"}}, "canonical_values": ["<canonical_value>"]}}
- set_null: {{}}
- none: {{}}

Infer params from sample values and non-conforming samples.
Do NOT assume a fixed format.

NUMERIC SUMMARY ANALYSIS:
If "numeric_summary" exists, use it.
Non-conforming samples detect format anomalies, not numerically implausible values.

Expected ranges:
- year: 1900-2100
- month_number: 1-12
- day_number: 1-31
- count: >= 0
- percentage: 0-100
- measure: reason from mean and median

If min or max fall outside the expected range:
- use "extract" only if the invalid value appears recoverable
- use "set_null" if it cannot be recovered
- never use "map" for numerically implausible values

COLUMN PROFILE:
{json.dumps(deterministic_profile, ensure_ascii=False, indent=2)}

Return ONLY a JSON object with EXACTLY these keys:
{{
  "role": "<role from vocabulary or custom:...>",
  "transformation_rule": {{...}},
  "description": "<1-2 sentences>"
}}
""".strip()


def _parse_llm_enrichment_response(response_text: str) -> dict:
    text = response_text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def _fallback_enrichment(deterministic_profile: dict, error: str) -> dict:
    return {
        "role": "unknown",
        "role_source": "fallback",
        "expected_dtype": "string",
        "transformation_rule": {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "Fallback: no transformation applied (LLM enrichment failed).",
        },
        "description": f"LLM enrichment failed: {error}",
    }


def _validate_and_fix_enrichment(
    enrichment: dict,
    deterministic_profile: dict,
) -> dict:
    fixes = []

    role = enrichment.get("role", "unknown")
    expected_dtype = enrichment.get("expected_dtype")
    rule = enrichment.get("transformation_rule") or {}
    action = rule.get("action", "none")
    params = rule.get("params") or {}

    dominant = (deterministic_profile.get("dominant_format") or {}).get("label")
    dominant_share = (deterministic_profile.get("dominant_format") or {}).get("share_pct", 0)

    NUMERIC_ROLES = {
        "identifier_numeric", "count", "measure", "percentage",
        "year", "month_number", "day_number",
    }
    if dominant in {"alpha", "generic"} and dominant_share >= 80 and role in NUMERIC_ROLES:
        n_unique = deterministic_profile.get("n_unique_non_null", 0)
        row_count = deterministic_profile.get("row_count", 1)
        unique_ratio = n_unique / max(row_count, 1)

        new_role = "category" if unique_ratio < 0.05 else "free_text"
        new_dtype = resolve_expected_dtype(new_role)

        fixes.append(
            f"role '{role}' is numeric but dominant_format is '{dominant}' at "
            f"{dominant_share}% (textual evidence overrides). "
            f"Reassigned to '{new_role}', dtype to '{new_dtype}'."
        )
        role = new_role
        expected_dtype = new_dtype
        action = "none"
        params = {}

    fixed_rule = {
        "action": action,
        "target_format": rule.get("target_format", dominant),
        "params": params,
        "description": rule.get("description", ""),
    }

    out = dict(enrichment)
    out["role"] = role
    out["expected_dtype"] = expected_dtype
    out["transformation_rule"] = fixed_rule
    if fixes:
        out["validator_fixes"] = fixes
    return out


def enrich_column_profile_with_llm(
    deterministic_profile: dict,
    llm_callable: Optional[Callable[[str], str]] = None,
) -> dict:
    if llm_callable is None:
        return _fallback_enrichment(deterministic_profile, "no llm_callable provided")

    prompt = _build_enrichment_prompt(deterministic_profile)

    try:
        raw = llm_callable(prompt)
        parsed = _parse_llm_enrichment_response(str(raw))
    except Exception as e:
        return _fallback_enrichment(deterministic_profile, str(e))

    # Validate role
    role = parsed.get("role", "unknown")
    if role in COLUMN_ROLES:
        role_source = "llm"
    elif isinstance(role, str) and role.startswith("custom:"):
        role_source = "custom"
    else:
        role = "unknown"
        role_source = "fallback"

    expected_dtype = resolve_expected_dtype(role)

    rule = parsed.get("transformation_rule") or {}
    if not isinstance(rule, dict) or rule.get("action") not in TRANSFORMATION_ACTIONS:
        rule = {
            "action": "none",
            "target_format": deterministic_profile.get("dominant_format", {}).get("label"),
            "params": {},
            "description": "LLM returned an invalid transformation_rule; defaulted to 'none'.",
        }
    rule.setdefault("params", {})
    rule.setdefault("target_format", deterministic_profile.get("dominant_format", {}).get("label"))
    rule.setdefault("description", "")

    enrichment = {
        "role": role,
        "role_source": role_source,
        "expected_dtype": expected_dtype,
        "transformation_rule": rule,
        "description": str(parsed.get("description", "")).strip(),
    }

    enrichment = _validate_and_fix_enrichment(enrichment, deterministic_profile)
    return enrichment

## Column Profiling & Cleaning Plan

This section builds the metadata that drives the cleaning pipeline.

- **`profile_single_column(df, column_name, llm_callable)`**  
  Creates a rich profile for one column by combining deterministic statistics with LLM-based enrichment.  
  It captures missing values, unique values, sample values, dominant format, non-conforming samples, numeric summary, semantic role, expected dtype, and transformation rule.

- **`profile_all_columns(df, llm_callable)`**  
  Applies `profile_single_column` to every column in the dataset.

- **`build_cleaning_plan(rich_profile_per_column)`**  
  Converts the full column profile into a slim cleaning plan.  
  This keeps only the information needed by the cleaning agent: expected dtype, dominant format, and transformation rule.

- **`build_dataset_column_profiles(...)`**  
  Main orchestration function.  
  It writes both:
  - the rich column profile JSON
  - the slim cleaning plan JSON

This step separates **profiling and decision-making** from the actual cleaning execution.

In [773]:
def profile_single_column(
    df: pd.DataFrame,
    column_name: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    series = df[column_name]
    non_missing = series.dropna()

    sample_values = [_to_native(v) for v in non_missing.head(5).tolist()]

    dominant = detect_dominant_format(series, sample_size=sample_size)
    non_conforming = collect_non_conforming_samples(
        series,
        dominant_label=dominant["label"],
        sample_size=sample_size,
        max_samples=100,
    )

    missing_stats = get_column_missing_stats(df[[column_name]])[column_name]

    deterministic_profile = {
        "column_name": column_name,
        "dtype": str(series.dtype),
        "row_count": int(len(series)),
        "null_count": missing_stats["null_count"],
        "non_null_count": missing_stats["non_null_count"],
        "missing_percentage": missing_stats["missing_percentage"],
        "n_unique_non_null": int(non_missing.nunique(dropna=True)),
        "unique_ratio_non_null": round(
            float(non_missing.nunique(dropna=True) / max(len(non_missing), 1)), 4
        ),
        "sample_values": sample_values,
        "dominant_format": dominant,
        "non_conforming": non_conforming,
    }

    numeric = _try_numeric(non_missing)
    if len(non_missing) > 0 and numeric.notna().mean() >= 0.9:
        numeric = numeric.dropna()
        if not numeric.empty:
            deterministic_profile["numeric_summary"] = {
                "min": _to_native(numeric.min()),
                "max": _to_native(numeric.max()),
                "mean": _to_native(round(float(numeric.mean()), 4)),
                "median": _to_native(round(float(numeric.median()), 4)),
                "std": _to_native(round(float(numeric.std(ddof=1)), 4)) if len(numeric) > 1 else None,
            }

    enrichment = enrich_column_profile_with_llm(
        deterministic_profile=deterministic_profile,
        llm_callable=llm_callable,
    )

    full_profile = {**deterministic_profile, **enrichment}
    return _to_native(full_profile)


def profile_all_columns(
    df: pd.DataFrame,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
) -> dict:
    return {
        col: profile_single_column(df, col, llm_callable=llm_callable, sample_size=sample_size)
        for col in df.columns
    }


def build_cleaning_plan(rich_profile_per_column: dict) -> dict:

    plan = {}
    for col_name, profile in rich_profile_per_column.items():
        dominant = profile.get("dominant_format", {}) or {}
        plan[col_name] = {
            "column_name": col_name,
            "expected_dtype": profile.get("expected_dtype", "string"),
            "dominant_format": dominant.get("label"),
            "transformation_rule": profile.get("transformation_rule", {
                "action": "none",
                "target_format": dominant.get("label"),
                "params": {},
                "description": "",
            }),
        }
    return plan


def build_dataset_column_profiles(
    df: pd.DataFrame,
    dataset_name: str,
    rich_json_path: str,
    slim_json_path: str,
    llm_callable: Optional[Callable[[str], str]] = None,
    sample_size: int = DOMINANT_FORMAT_SAMPLE_SIZE,
    findings_root_key: str = "column_profiles",
) -> dict:

    findings = _safe_load_json(rich_json_path)

    dataset_summary = {
        "dataset_name": dataset_name,
        "shape": get_dataframe_shape(df),
        "dtypes": get_dataframe_dtypes(df),
    }

    columns_payload = profile_all_columns(
        df,
        llm_callable=llm_callable,
        sample_size=sample_size,
    )

    findings.setdefault(findings_root_key, {})
    findings[findings_root_key][dataset_name] = {
        "dataset_summary": dataset_summary,
        "columns": columns_payload,
    }
    _safe_write_json(rich_json_path, findings)

    slim_all = _safe_load_json(slim_json_path)
    slim_all[dataset_name] = build_cleaning_plan(columns_payload)
    _safe_write_json(slim_json_path, slim_all)

    return findings[findings_root_key][dataset_name]

## Cleaning Rule Application

This section applies the cleaning rules generated during profiling to each dataset.

- **Action masks**  
  Each transformation action defines its own target rows:
  - `map` applies only to values found in the mapping dictionary
  - `extract` applies to values that do not already match the expected regex
  - `set_null` targets values that do not match the dominant format
  - `none` leaves the column unchanged

- **Action implementations**  
  The dispatcher applies the correct transformation rule to each column based on the cleaning plan.

- **`enforce_expected_dtype(series, expected_dtype)`**  
  Converts each cleaned column to its expected pandas dtype, such as nullable integers, floats, strings, or datetimes.

- **`clean_dataset(...)`**  
  Main cleaning function. It:
  1. loads the raw CSV
  2. normalizes column names and text values
  3. standardizes missing values
  4. applies profile-driven transformation rules
  5. enforces expected data types
  6. removes duplicate rows
  7. writes the cleaned dataset
  8. returns a structured cleaning summary

This is the execution layer of the cleaning pipeline: the LLM defines the rules, while deterministic code applies them safely.

In [774]:
def _is_missing(v) -> bool:
    return v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA


def _smart_recover(value, dominant_label: Optional[str]):

    if _is_missing(value):
        return value

    s = str(value).strip()
    if s == "":
        return pd.NA

    if dominant_label == "digit":
        if "-" in s:
            return pd.NA
        m = re.search(r"\d+", s)
        return m.group(0) if m else pd.NA

    if dominant_label == "float":
        m = re.search(r"-?\d+[.,]?\d*", s)
        if not m:
            return pd.NA
        return m.group(0).replace(",", ".")

    if dominant_label == "alpha":
        m = re.search(r"[A-Za-z]+", s)
        return m.group(0) if m else pd.NA

    if dominant_label == "alphanumeric":
        m = re.search(r"[A-Za-z0-9]+", s)
        return m.group(0) if m else pd.NA

    return pd.NA


def _mask_for_map(series: pd.Series, params: dict) -> pd.Series:
    mapping = params.get("mapping", {}) or {}
    if not mapping:
        return pd.Series(False, index=series.index)
    keys = {str(k).strip().lower() for k in mapping.keys()}

    def _hit(v):
        if _is_missing(v):
            return False
        return str(v).strip().lower() in keys

    return series.map(_hit).astype(bool)


def _mask_for_extract(series: pd.Series, params: dict) -> pd.Series:
    pattern = params.get("regex", r"\d+")
    full = re.compile(f"^(?:{pattern})$")

    def _needs_extract(v):
        if _is_missing(v):
            return False
        return full.match(str(v).strip()) is None

    return series.map(_needs_extract).astype(bool)


def _mask_for_set_null(series: pd.Series, dominant_label: Optional[str]) -> pd.Series:
    if dominant_label is None:
        return series.notna()  

    def _non_conforming(v):
        if _is_missing(v):
            return False
        return _value_format_label(v) != dominant_label

    return series.map(_non_conforming).astype(bool)


def _action_none(series: pd.Series, params: dict) -> pd.Series:
    return series


def _action_extract(series: pd.Series, params: dict) -> pd.Series:
    pattern = params.get("regex", r"\d+")

    def _extract_one(v):
        if _is_missing(v):
            return v

        m = re.search(pattern, str(v).strip())
        if not m:
            return pd.NA

        if m.groups():
            return next((g for g in m.groups() if g is not None), pd.NA)

        return m.group(0)

    return series.map(_extract_one)


def _action_map(series: pd.Series, params: dict) -> pd.Series:
    mapping = params.get("mapping", {}) or {}
    if not mapping:
        return series
    norm = {str(k).strip().lower(): (pd.NA if v is None else str(v))
            for k, v in mapping.items()}

    def _map_one(v):
        if _is_missing(v):
            return v
        return norm.get(str(v).strip().lower(), v)

    return series.map(_map_one)


def _action_set_null(series: pd.Series, params: dict, dominant_label: Optional[str]) -> pd.Series:
    return series.map(lambda v: _smart_recover(v, dominant_label))


ACTION_DISPATCHER = {
    "none":     _action_none,
    "extract":  _action_extract,
    "map":      _action_map,
    "set_null": _action_set_null,
}


def apply_transformation_rule(
    series: pd.Series,
    transformation_rule: dict,
    dominant_label: Optional[str],
) -> pd.Series:
    action = (transformation_rule or {}).get("action", "none")
    params = (transformation_rule or {}).get("params", {}) or {}

    if action == "none" or action not in ACTION_DISPATCHER:
        return series

    if action == "map":
        target_mask = _mask_for_map(series, params)
    elif action == "extract":
        target_mask = _mask_for_extract(series, params)
    elif action == "set_null":
        target_mask = _mask_for_set_null(series, dominant_label)
    else:
        target_mask = pd.Series(False, index=series.index)

    if not target_mask.any():
        return series

    out = series.astype("object").copy()
    sub = out.loc[target_mask]

    if action == "set_null":
        out.loc[target_mask] = _action_set_null(sub, params, dominant_label)
    else:
        out.loc[target_mask] = ACTION_DISPATCHER[action](sub, params)

    return out


def enforce_expected_dtype(series: pd.Series, expected_dtype: str) -> pd.Series:
    if expected_dtype is None or str(series.dtype) == expected_dtype:
        return series

    try:
        if expected_dtype.startswith("datetime"):
            return pd.to_datetime(series, errors="coerce")
        if expected_dtype in {"Int64", "Int32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        if expected_dtype in {"Float64", "Float32", "float64", "float32"}:
            return pd.to_numeric(series, errors="coerce").astype(expected_dtype)
        return series.astype(expected_dtype)
    except Exception:
        return series


def clean_dataset(
    input_csv_path: str,
    output_csv_path: str,
    cleaning_plan_path: str,
    dataset_key: str,
    read_csv_kwargs: Optional[dict] = None,
) -> dict:
    read_csv_kwargs = read_csv_kwargs or {}

    df = pd.read_csv(input_csv_path, **read_csv_kwargs)
    shape_before = df.shape

    df = normalize_column_names(df)
    df = deduplicate_column_names(df)
    df = standardize_missing_values(df)
    df = normalize_text_values(df)

    plan = load_cleaning_plan(cleaning_plan_path, dataset_key)

    per_column_actions = {}
    dtype_changes = {}

    for col in df.columns:
        col_plan = plan.get(col)
        if not col_plan:
            per_column_actions[col] = {"action": "skipped (no plan)"}
            continue

        rule = col_plan.get("transformation_rule", {}) or {}
        dominant_label = col_plan.get("dominant_format")
        expected_dtype = col_plan.get("expected_dtype")

        original = df[col]
        df[col] = apply_transformation_rule(
            series=df[col],
            transformation_rule=rule,
            dominant_label=dominant_label,
        )

        dtype_before = str(df[col].dtype)
        df[col] = enforce_expected_dtype(df[col], expected_dtype)
        dtype_after = str(df[col].dtype)

        n_changed = int((original.astype("string") != df[col].astype("string")).sum())
        per_column_actions[col] = {
            "action": rule.get("action", "none"),
            "rows_changed": n_changed,
            "dtype_before": dtype_before,
            "dtype_after": dtype_after,
            "expected_dtype": expected_dtype,
            "dtype_match": (dtype_after == expected_dtype) if expected_dtype else None,
        }
        if dtype_before != dtype_after:
            dtype_changes[col] = {"before": dtype_before, "after": dtype_after}

    df = remove_duplicate_rows(df)

    print(f"DTYPE DIAGNOSTIC for {dataset_key}  (pre-to_csv)")
    print(f"{'column':<30} {'expected':<20} {'actual':<20} OK?")
    for col, info in per_column_actions.items():
        if info.get('action') == 'skipped (no plan)':
            continue
        exp = info.get('expected_dtype') or '(none)'
        act = info.get('dtype_after') or '?'
        ok = 'OK' if (info.get('expected_dtype') is None or info.get('dtype_match')) else 'NO'
        print(f"{col:<30} {exp:<20} {act:<20} {ok}")

    shape_after = df.shape
    os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
    df.to_csv(output_csv_path, index=False)

    return {
        "dataset_key": dataset_key,
        "input_path": input_csv_path,
        "output_path": output_csv_path,
        "shape_before": list(shape_before),
        "shape_after": list(shape_after),
        "duplicate_rows_removed": int(shape_before[0] - shape_after[0]),
        "per_column_actions": per_column_actions,
        "dtype_changes": dtype_changes,
    }

## LLM Interface & Pre-Pipeline Profiling

This section connects the LLM to the pipeline and runs column profiling on all input datasets.

- **`llm_callable(prompt)`**  
  Wrapper around the LLM API.  
  Sends the prompt and enforces a strict JSON-only response format to ensure consistency and easy parsing.

- **`run_pre_pipeline_column_profiling(datasets)`**  
  Executes the profiling phase for multiple datasets. It:
  1. loads each dataset
  2. applies basic normalization (column names, missing values, text)
  3. runs column profiling (deterministic + LLM enrichment)
  4. saves both rich profiles and cleaning plans

This step prepares all datasets for the cleaning phase by generating the metadata required to drive transformations.

In [775]:
def llm_callable(prompt: str) -> str:
    response = _client.chat.completions.create(
        model=MISTRAL_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data profiling assistant. "
                    "When asked, you return ONLY a valid JSON object — "
                    "no preamble, no markdown fences, no explanations outside the JSON. "
                    "Do not invent facts not supported by the provided profile."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=1024,
        response_format={"type": "json_object"},
    )
    return response.choices[0].message.content.strip()

def run_pre_pipeline_column_profiling(
    datasets: list[tuple[str, str]],
) -> dict:
    profiles = {}

    for dataset_name, csv_path in datasets:
        if not os.path.exists(csv_path):
            print(f"  [profiling] File not found, skipped: {csv_path}")
            continue

        try:
            original_cols = list(pd.read_csv(csv_path, nrows=0).columns)
            print(f"  [profiling] Running column profiling for {dataset_name} "
                  f"({len(original_cols)} columns: {', '.join(original_cols[:5])}"
                  f"{'...' if len(original_cols) > 5 else ''})")
        except Exception:
            print(f"  [profiling] Running column profiling for {dataset_name}...")

        df = pd.read_csv(csv_path)
        df = normalize_column_names(df)
        df = deduplicate_column_names(df)
        df = standardize_missing_values(df)
        df = normalize_text_values(df)

        profiles[dataset_name] = build_dataset_column_profiles(
            df=df,
            dataset_name=dataset_name,
            rich_json_path=COLUMN_PROFILES_JSON,
            slim_json_path=CLEANING_PLAN_JSON,
            llm_callable=llm_callable,
            sample_size=DOMINANT_FORMAT_SAMPLE_SIZE,
        )
        print(f"  [profiling] Saved profile for {dataset_name}")

    return profiles


## Run: Column Profiling

This cell executes the column profiling step.

- If profiling results already exist (cached JSON files), it skips execution.
- Otherwise, it runs profiling on all input datasets and generates:
  - the rich column profiles
  - the slim cleaning plans

At the end, it confirms whether the output files were successfully created.

In [776]:
print("  PRE-PIPELINE COLUMN PROFILING")

if os.path.exists(COLUMN_PROFILES_JSON) and os.path.exists(CLEANING_PLAN_JSON):
    print(f"  [cache] Profiles already exist, skipping profiling.")
    print(f"   {COLUMN_PROFILES_JSON} ({os.path.getsize(COLUMN_PROFILES_JSON):,} bytes)")
    print(f"   {CLEANING_PLAN_JSON} ({os.path.getsize(CLEANING_PLAN_JSON):,} bytes)")
else:
    profiles = run_pre_pipeline_column_profiling(
        datasets=[
            ("allarmi_raw",    ALLARMI_CSV),
            ("tipologia_raw",  TIPOLOGIA_CSV),
        ]
    )

    print()
    print("Files written:")
    if os.path.exists(COLUMN_PROFILES_JSON):
        print(f"   {COLUMN_PROFILES_JSON} ({os.path.getsize(COLUMN_PROFILES_JSON):,} bytes)")
    else:
        print(f"   {COLUMN_PROFILES_JSON} (NOT created)")

    if os.path.exists(CLEANING_PLAN_JSON):
        print(f"   {CLEANING_PLAN_JSON} ({os.path.getsize(CLEANING_PLAN_JSON):,} bytes)")
    else:
        print(f"   {CLEANING_PLAN_JSON} (NOT created)")

  PRE-PIPELINE COLUMN PROFILING
  [profiling] Running column profiling for allarmi_raw (24 columns: OCCORRENZE, AREOPORTO_ARRIVO, AREOPORTO_PARTENZA, ANNO_PARTENZA, MESE_PARTENZA...)
  [profiling] Saved profile for allarmi_raw
  [profiling] Running column profiling for tipologia_raw (33 columns: NAZIONALITA, AREOPORTO_ARRIVO, AREOPORTO_PARTENZA, ANNO_PARTENZA, MESE_PARTENZA...)
  [profiling] Saved profile for tipologia_raw

Files written:
   /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/column_profiles.json (83,881 bytes)
   /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/cleaning_plan.json (25,494 bytes)


# Cleaning Agent

This cell executes the data cleaning step for all datasets.

- It loops over each dataset and checks:
  - if the input file exists
  - if a cleaned version is already available (cache)

- For each dataset, it:
  1. applies the cleaning plan generated during profiling
  2. transforms values according to LLM-defined rules
  3. enforces expected data types
  4. removes duplicate rows
  5. saves the cleaned dataset

- After processing, it prints a summary including:
  - shape before and after cleaning
  - number of duplicates removed
  - dtype changes

This is the execution phase where all transformations are applied to produce the final cleaned datasets.

In [777]:
print("  CLEANING AGENT")

datasets_to_clean = [
    ("allarmi_raw",   ALLARMI_CSV,   os.path.join(OUTPUT_DIR, "allarmi_clean.csv")),
    ("tipologia_raw", TIPOLOGIA_CSV, os.path.join(OUTPUT_DIR, "tipologia_clean.csv")),
]

for dataset_key, input_path, output_path in datasets_to_clean:
    if not os.path.exists(input_path):
        print(f"  [cleaning] File not found, skipped: {input_path}")
        continue

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"  [cache] '{dataset_key}' already cleaned  {os.path.basename(output_path)} ({os.path.getsize(output_path):,} bytes), skipping.")
        continue

    print(f"  [cleaning] Processing '{dataset_key}'...")
    report = clean_dataset(
        input_csv_path=input_path,
        output_csv_path=output_path,
        cleaning_plan_path=CLEANING_PLAN_JSON,
        dataset_key=dataset_key,
    )

    print(f"  [cleaning] Shape: {report['shape_before']}  {report['shape_after']}")
    print(f"  [cleaning] Duplicate rows removed: {report['duplicate_rows_removed']}")
    print(f"  [cleaning] Dtype changes: {report['dtype_changes']}")
    print(f"   Saved: {output_path}")
    print()

  CLEANING AGENT
  [cleaning] Processing 'allarmi_raw'...
DTYPE DIAGNOSTIC for allarmi_raw  (pre-to_csv)
column                         expected             actual               OK?
occorrenze                     string               string               OK
areoporto_arrivo               string               string               OK
areoporto_partenza             string               string               OK
anno_partenza                  Int64                Int64                OK
mese_partenza                  Int64                Int64                OK
data_partenza                  datetime64[ns]       datetime64[us]       NO
descr_aereoporto_arr           string               string               OK
descr_aereoporto_part          string               string               OK
citta_arr                      string               string               OK
citta_partenza                 string               string               OK
codice_paese_arr               string               string

## Findings Helper

This helper generates prompt instructions for agents to read and update a shared findings JSON file.

- **`_findings_guidance(task_key, extra_notes)`**  
  Returns a standardized instruction string that tells each agent how to:
  1. safely load the shared findings file (handling missing or corrupted cases)
  2. write its results under a specific task key
  3. preserve information written by other agents
  4. store only JSON-serializable data (no pandas/numpy objects)

This ensures consistent communication and state sharing across all agents in the pipeline.

In [778]:
def _findings_guidance(task_key: str, extra_notes: str = "") -> str:
    base = (
        f"Maintain a shared findings JSON at '{FINDINGS_JSON}'. "
        f"At the start, attempt to load it; if missing, empty, invalid, or corrupted, "
        f"initialize an empty dict instead of failing. "
        f"Store new information under the key '{task_key}' while preserving existing keys "
        f"for other tasks. "
        f"Use concise, machine-readable fields with only native Python JSON-serializable types "
        f"such as dict, list, str, int, float, bool, or null. "
        f"Convert pandas and numpy values to native Python types before saving. "
        f"Convert tuples to lists before saving. "
        f"After completing the task, update the entry for '{task_key}' and write the full JSON "
        f"back by overwriting the file. "
    )
    if extra_notes:
        base += extra_notes
    return base

## Code Executor

This component handles the execution of code generated by the LLM.

- **System Prompt**  
  Defines strict rules for the code-generation agent:
  - output must be pure Python code
  - no explanations or markdown
  - only safe, local libraries allowed
  - must print progress and a final success message

- **`_extract_code_from_response(raw)`**  
  Cleans the LLM response by removing markdown fences or handling truncated outputs, ensuring valid executable code is extracted.

- **`code_executor_run(prompt, ...)`**  
  Core execution loop:
  1. sends a prompt to the LLM
  2. extracts the generated Python code
  3. executes it in a controlled environment
  4. captures stdout and stderr
  5. returns execution results (code, logs, success flag)

This module acts as the runtime engine of the agentic system, turning LLM-generated instructions into actual data processing steps.

In [779]:
CODE_EXECUTOR_SYSTEM_PROMPT = (
    "You are a code-generation agent. "
    "When given a task, you respond ONLY with executable Python code — "
    "no preamble, no explanations, no markdown fences. "
    "Use only standard libraries plus pandas, numpy, json, os, re, math. "
    "Do not import subprocess, requests, or any networking module. "
    "All inputs and outputs are file paths on the local filesystem. "
    "Always print short progress messages so the orchestrator can see what happened. "
    "Always end with a clear print line confirming success or raising an explicit error."
)


def _extract_code_from_response(raw: str) -> str:
    s = raw.strip()

    m = re.search(r"```(?:python)?\s*(.*?)```", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    m = re.search(r"```(?:python)?\s*\n(.*)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    return s


def code_executor_run(
    prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 4096,
) -> dict:

    response = _client.chat.completions.create(
        model=MISTRAL_MODEL,
        messages=[
            {"role": "system", "content": CODE_EXECUTOR_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    raw = response.choices[0].message.content
    code = _extract_code_from_response(raw)

    captured_stdout = io.StringIO()
    captured_stderr = io.StringIO()
    success = True

    old_stdout, old_stderr = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = captured_stdout, captured_stderr

    try:
        exec(code, {"__name__": "__agent__"})
    except Exception:
        success = False
        captured_stderr.write(traceback.format_exc())
    finally:
        sys.stdout, sys.stderr = old_stdout, old_stderr

    return {
        "code":    code,
        "stdout":  captured_stdout.getvalue(),
        "stderr":  captured_stderr.getvalue(),
        "success": success,
    }

## Validators

This section defines deterministic validators for each artifact produced by the agents.

Each validator checks that the expected output file exists, can be loaded, contains the required columns or sections, and satisfies basic consistency rules.

- **`validate_baseline_output`**  
  Validates `baseline_data.csv`.

- **`validate_outlier_output`**  
  Validates `outliers.csv`.

- **`validate_risk_output`**  
  Validates `risk_report.csv`.

- **`validate_report_output`**  
  Validates the final markdown report.

- **`print_validation_report`**  
  Prints validator results in a readable format.

These checks are deterministic and do not use the LLM, ensuring that each pipeline step produces a valid artifact before moving to the next agent.

In [780]:
# Baseline Validator

def validate_baseline_output(output_csv_path: str) -> dict:
    """
    Validates the dataframe produced by the Baseline Agent.
    """

    checks = []

    file_exists = os.path.exists(output_csv_path)
    checks.append({
        "name":   "file_exists",
        "ok":     file_exists,
        "detail": output_csv_path,
    })
    if not file_exists:
        return {"passed": False, "checks": checks}

    try:
        df = pd.read_csv(output_csv_path)
    except Exception as e:
        checks.append({"name": "loadable", "ok": False, "detail": str(e)})
        return {"passed": False, "checks": checks}

    checks.append({"name": "loadable", "ok": True, "detail": f"shape={df.shape}"})

    required = [
        "route",
        "rolling_mean_alarms",
        "rolling_std_alarms",
        "total_alarms",
        "n_records",
        "alert_rate",
    ]

    missing = [c for c in required if c not in df.columns]

    checks.append({
        "name":   "required_columns",
        "ok":     len(missing) == 0,
        "detail": f"missing={missing}" if missing else f"all present: {required}",
    })

    checks.append({
        "name":   "non_empty",
        "ok":     len(df) > 0,
        "detail": f"rows={len(df)}",
    })

    if len(missing) == 0:

        numeric_cols = [
            "rolling_mean_alarms",
            "rolling_std_alarms",
            "total_alarms",
            "n_records",
            "alert_rate",
        ]

        non_numeric = [
            c for c in numeric_cols
            if not pd.api.types.is_numeric_dtype(df[c])
        ]

        checks.append({
            "name":   "numeric_dtypes",
            "ok":     len(non_numeric) == 0,
            "detail": f"non_numeric={non_numeric}" if non_numeric else "all numeric",
        })

        inf_cols = [
            c for c in numeric_cols
            if np.isinf(df[c].fillna(0)).any()
        ]

        checks.append({
            "name":   "no_infinity",
            "ok":     len(inf_cols) == 0,
            "detail": f"inf_in={inf_cols}" if inf_cols else "no inf",
        })

        route_ok = (
            df["route"].notna().all()
            and (df["route"].astype(str).str.strip() != "").all()
        )

        checks.append({
            "name":   "route_populated",
            "ok":     route_ok,
            "detail": "all rows have a route" if route_ok else "some routes are empty",
        })

    passed = all(c["ok"] for c in checks)
    return {"passed": passed, "checks": checks}


# Outlier Detection Validator

def validate_outlier_output(output_csv_path: str) -> dict:
    checks = []

    file_exists = os.path.exists(output_csv_path)
    checks.append({"name": "file_exists", "ok": file_exists, "detail": output_csv_path})
    if not file_exists:
        return {"passed": False, "checks": checks}

    try:
        df = pd.read_csv(output_csv_path)
    except Exception as e:
        checks.append({"name": "loadable", "ok": False, "detail": str(e)})
        return {"passed": False, "checks": checks}

    checks.append({"name": "loadable", "ok": True, "detail": f"shape={df.shape}"})

    required = ["route", "anomaly_score", "is_outlier"]
    missing = [c for c in required if c not in df.columns]

    checks.append({
        "name": "required_columns",
        "ok": len(missing) == 0,
        "detail": f"missing={missing}" if missing else f"all present: {required}",
    })

    checks.append({
        "name": "non_empty",
        "ok": len(df) > 0,
        "detail": f"rows={len(df)}",
    })

    if "is_outlier" in df.columns:
        checks.append({
            "name": "outliers_flagged",
            "ok": df["is_outlier"].astype(bool).sum() > 0,
            "detail": f"outliers={df['is_outlier'].astype(bool).sum()}",
        })

    passed = all(c["ok"] for c in checks)
    return {"passed": passed, "checks": checks}


# Risk Profiling Validator

def validate_risk_output(path):
    checks = []

    if not os.path.exists(path):
        return {"passed": False, "checks": [{"name": "file_exists", "ok": False, "detail": path}]}

    df = pd.read_csv(path)

    required = ["route", "risk_score", "risk_level", "risk_reason"]
    missing = [c for c in required if c not in df.columns]

    checks.append({
        "name": "required_columns",
        "ok": len(missing) == 0,
        "detail": missing if missing else "ok",
    })

    checks.append({
        "name": "non_empty",
        "ok": len(df) > 0,
        "detail": len(df),
    })

    checks.append({
        "name": "valid_levels",
        "ok": df["risk_level"].isin(["LOW", "MEDIUM", "HIGH"]).all(),
        "detail": df["risk_level"].unique(),
    })

    return {"passed": all(c["ok"] for c in checks), "checks": checks}


# Report Validator

def validate_report_output(path):
    checks = []

    file_exists = os.path.exists(path)
    checks.append({"name": "file_exists", "ok": file_exists, "detail": path})
    if not file_exists:
        return {"passed": False, "checks": checks}

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    required_sections = [
        "Executive Summary",
        "Risk Distribution",
        "Top High-Risk Routes",
        "Main Drivers",
        "Recommended Next Actions",
    ]

    missing = [s for s in required_sections if s not in text]

    checks.append({
        "name": "required_sections",
        "ok": len(missing) == 0,
        "detail": f"missing={missing}" if missing else "all present",
    })

    checks.append({
        "name": "non_empty",
        "ok": len(text.strip()) > 200,
        "detail": f"chars={len(text)}",
    })

    return {"passed": all(c["ok"] for c in checks), "checks": checks}

## Supervisor

This section defines the deterministic supervisor that runs each agent.

- It sends the agent prompt to the Code Executor.
- It captures the generated code output.
- It validates the produced artifact using the correct validator.
- If execution or validation fails, it retries with a higher temperature.
- It returns a structured summary of the run, including success status, attempts, logs, generated code, and validation result.

The supervisor is the orchestration layer that makes each agent execution reliable and inspectable.

In [781]:
def run_agent_with_supervisor(
    task_name: str,
    prompt: str,
    validator_fn,             
    output_path: str,
    max_retries: int = 3,
    retry_temperatures: tuple = (0.0, 0.3, 0.6, 0.9),
) -> dict:

    print(f"  AGENT: {task_name}")

    last_exec = None
    last_verdict = None

    for attempt_idx in range(max_retries + 1):
        temp = retry_temperatures[min(attempt_idx, len(retry_temperatures) - 1)]
        print(f"\n  [attempt {attempt_idx + 1}/{max_retries + 1}] temperature={temp}")

        last_exec = code_executor_run(prompt, temperature=temp)

        if last_exec["stdout"]:
            print("  ── stdout ──")
            for line in last_exec["stdout"].rstrip().splitlines():
                print(f"  │ {line}")

        if not last_exec["success"]:
            print("  Execution Failed")
            tail = last_exec["stderr"].rstrip().splitlines()[-8:]
            for line in tail:
                print(f"  ! {line}")
            continue

        last_verdict = validator_fn(output_path)
        print_validation_report(last_verdict, label=f"attempt {attempt_idx + 1}")

        if last_verdict["passed"]:
            print(f"   {task_name} SUCCEEDED on attempt {attempt_idx + 1}")
            return {
                "task":       task_name,
                "succeeded":  True,
                "attempts":   attempt_idx + 1,
                "last_code":  last_exec["code"],
                "last_stdout": last_exec["stdout"],
                "last_stderr": last_exec["stderr"],
                "last_validation": last_verdict,
            }

    print(f"\n   {task_name} FAILED after {max_retries + 1} attempts")
    return {
        "task":       task_name,
        "succeeded":  False,
        "attempts":   max_retries + 1,
        "last_code":  last_exec["code"] if last_exec else "",
        "last_stdout": last_exec["stdout"] if last_exec else "",
        "last_stderr": last_exec["stderr"] if last_exec else "",
        "last_validation": last_verdict,
    }

## Baseline Agent

This agent builds the route-level baseline dataset used for anomaly detection.

- It loads the cleaned datasets (`allarmi_clean.csv`, `tipologia_clean.csv`)
- Selects the dataset containing a meaningful alarm-related numeric column
- Identifies departure and arrival columns
- Constructs a `route` feature (`departure→arrival`)
- Aggregates data per route to compute:
  - total alarms
  - number of records
  - alert rate
- Computes baseline statistics (`rolling_mean_alarms`, `rolling_std_alarms`)
- Ensures correct data types and handles invalid values
- Writes the final `baseline_data.csv`

This step creates the foundational dataset on which outlier detection is performed.

In [782]:
def _build_baseline_prompt():
    return (
        "You are the Baseline Agent. "
        "Your task is to build a route-level baseline dataset from cleaned input data. "

        f"Inputs: '{OUTPUT_DIR}/allarmi_clean.csv' and '{OUTPUT_DIR}/tipologia_clean.csv'. "
        f"Output: '{OUTPUT_DIR}/baseline_data.csv'. "

        "Generate Python code using pandas, numpy, json, and os. "
        "Do not require user input. "

        "STEP 1 — LOAD DATA: "
        "Load both datasets and print their shapes. "

        "STEP 2 — SELECT DATASET: "
        "Choose the dataset that contains a meaningful alarm-related numeric column. "
        "Prefer columns such as 'allarmati', 'tot', or similar. "
        "If multiple candidates exist, select the most relevant one. "
        "Raise ValueError if no numeric measure can be identified. "

        "STEP 3 — IDENTIFY COLUMNS: "
        "Identify departure and arrival columns. "
        "Typical names include 'areoporto_partenza' and 'areoporto_arrivo'. "
        "Raise ValueError if they cannot be found. "

        "STEP 4 — BUILD ROUTE: "
        "Create a 'route' column as 'departure→arrival'. "
        "Ensure no missing or empty routes. "

        "STEP 5 — AGGREGATE: "
        "Group by route and compute: "
        "- total_alarms = sum of the selected measure; "
        "- n_records = number of rows per route; "
        "- alert_rate = total_alarms / n_records. "

        "STEP 6 — BASELINE FEATURES: "
        "Compute rolling_mean_alarms and rolling_std_alarms. "
        "If no time dimension is available, compute mean and std directly per route. "
        "If std is NaN, leave it as NaN. "

        "STEP 7 — CLEAN OUTPUT: "
        "Ensure numeric columns are numeric. "
        "Replace infinite values with NaN and handle safely. "

        "STEP 8 — OUTPUT: "
        "Write baseline_data.csv with columns: "
        "route, rolling_mean_alarms, rolling_std_alarms, "
        "total_alarms, n_records, alert_rate. "

        "STEP 9 — SUMMARY: "
        "Print number of routes and preview the dataset. "
    )

## Run: Baseline Agent

This cell executes the Baseline Agent through the supervisor.

- The supervisor:
  - generates code via the LLM
  - executes it
  - validates the output
  - retries if needed

- If successful, it:
  - loads the generated `baseline_data.csv`
  - prints its shape
  - shows a preview of the dataset

This step produces the baseline dataset used by the Outlier Detection Agent.

In [ ]:
baseline_output_csv = os.path.join(OUTPUT_DIR, "baseline_data.csv")

result = run_agent_with_supervisor(
    task_name="baseline",
    prompt=_build_baseline_prompt(),
    validator_fn=validate_baseline_output,
    output_path=baseline_output_csv,
    max_retries=3,
)

if result["succeeded"]:
    df_baseline = pd.read_csv(baseline_output_csv)
    print(f"\nbaseline_data.csv: shape={df_baseline.shape}")
    print(df_baseline.head())

  AGENT: baseline

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading data...
  │ allarmi_clean.csv shape: (5028, 24)
  │ tipologia_clean.csv shape: (5034, 33)
  │ Selecting dataset with meaningful alarm-related numeric column...
  │ Selected measure: tot
  │ Identifying departure and arrival columns...
  │ Departure column: areoporto_partenza, Arrival column: areoporto_arrivo
  │ Building route column...
  │ Number of valid routes: 368
  │ Aggregating data by route...
  │ Aggregated data shape: (368, 4)
  │ Computing baseline features...
  │ Cleaning output...
  │ Writing baseline_data.csv...
  │ Output written to /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/baseline_data.csv
  │ Number of routes: 368
  │ Preview of baseline_data.csv:
  │      route  total_alarms  n_records  alert_rate  rolling_mean_alarms  \
  │ 0  adb→mxp         291.0          6   48.500000                291.0   
  │ 1  add→fco        3138.0         29  108

## Outlier Detection Agent

This agent identifies anomalous routes using the baseline dataset.

- It loads `baseline_data.csv`
- Computes key anomaly signals:
  - **z_score** on total alarms
  - **ratio_to_baseline** comparing route alarms to baseline mean
- Builds a combined **anomaly_score**
- Selects the top 5% most anomalous routes
- Flags them as outliers (`is_outlier = True`)
- Writes the result to `outliers.csv`

This step isolates the most abnormal routes for further risk analysis.

In [ ]:
def _build_outlier_prompt():
    return (
        "You are the Outlier Detection Agent. "
        "Your task is to detect anomalous routes using the baseline dataset. "

        f"Input: '{OUTPUT_DIR}/baseline_data.csv'. "
        f"Output: '{OUTPUT_DIR}/outliers.csv'. "

        "Generate Python code using pandas, numpy, sklearn if available. "

        "STEP 1 — LOAD DATA: "
        "Load baseline_data.csv. "

        "STEP 2 — FEATURE ENGINEERING: "
        "Compute z_score using total_alarms: "
        "mean = df['total_alarms'].mean() "
        "std = df['total_alarms'].std() "
        "if std == 0 or NaN, set std = 1 "
        "df['z_score'] = (df['total_alarms'] - mean) / std "

        "Compute ratio_to_baseline: "
        "df['ratio_to_baseline'] = df['total_alarms'] / df['rolling_mean_alarms'] "

        "Replace inf with NaN. "

        "STEP 3 — ANOMALY SCORE: "
        "df['anomaly_score'] = "
        "  3 * df['ratio_to_baseline'].fillna(0) + "
        "  2 * df['total_alarms'].fillna(0) + "
        "  df['z_score'].abs().fillna(0) "

        "STEP 4 — TOP-K SELECTION: "
        "k = max(1, int(0.05 * len(df))) "
        "ranked = df.sort_values('anomaly_score', ascending=False) "
        "top_idx = ranked.head(k).index "

        "df['is_outlier'] = False "
        "df.loc[top_idx, 'is_outlier'] = True "

        "STEP 5 — OUTPUT: "
        "outliers = df[df['is_outlier'] == True].copy() "
        "Write outliers.csv. "

        "IMPORTANT: "
        "Always return at least one outlier. "
    )

## Run: Outlier Detection Agent

This cell runs the Outlier Detection Agent through the supervisor.

- The supervisor:
  - generates and executes code from the LLM
  - validates the output (`outliers.csv`)
  - retries if necessary

- If successful, it:
  - loads the detected outliers
  - prints dataset shape
  - displays the top anomalous routes

This step produces the subset of routes flagged as anomalies for risk profiling.

In [ ]:
outlier_output_csv = os.path.join(OUTPUT_DIR, "outliers.csv")

result_outlier = run_agent_with_supervisor(
    task_name="outlier_detection",
    prompt=_build_outlier_prompt(),
    validator_fn=validate_outlier_output,
    output_path=outlier_output_csv,
    max_retries=3,
)

if result_outlier["succeeded"]:
    df_outliers = pd.read_csv(outlier_output_csv)
    print(f"\noutliers.csv: shape={df_outliers.shape}")
    print(df_outliers.head(10))

  AGENT: outlier_detection

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading baseline_data.csv...
  │ Data loaded.
  │ Computing z_score...
  │ Computing ratio_to_baseline...
  │ Computing anomaly_score...
  │ Selecting top outliers...
  │ Writing outliers.csv...
  │ Outliers saved to outliers.csv.
  │ Outlier detection completed successfully.

  ┌─ Validation report attempt 1 ─
  │ ✓  file_exists            /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/outliers.csv
  │ ✓  loadable               shape=(18, 10)
  │ ✓  required_columns       all present: ['route', 'anomaly_score', 'is_outlier']
  │ ✓  non_empty              rows=18
  │ ✓  outliers_flagged       outliers=18
  └─ Overall: PASSED

   outlier_detection SUCCEEDED on attempt 1

outliers.csv: shape=(18, 10)
     route  total_alarms  n_records   alert_rate  rolling_mean_alarms  \
0  add→fco        3138.0         29   108.206897               3138.0   
1  auh→fco        3

## Risk Profiling Agent

This agent assigns a risk level to each anomalous route.

- It loads `outliers.csv`
- Validates required anomaly-related columns
- Normalizes the anomaly score into a **risk_score (0–100)**
- Assigns a **risk level**:
  - HIGH
  - MEDIUM
  - LOW
- Generates a human-readable **risk_reason** based on key signals
- Sorts routes by risk severity
- Writes the final `risk_report.csv`

This step translates anomalies into actionable risk categories for decision-making.

In [786]:
def _build_risk_prompt():
    return (
        "You are the Risk Profiling Agent. "
        "Your task is to assign a risk level to each anomalous route. "

        f"Input: '{OUTPUT_DIR}/outliers.csv'. "
        f"Output: '{OUTPUT_DIR}/risk_report.csv'. "

        "Generate Python code using pandas, numpy. "

        "STEP 1 — LOAD DATA: "
        "Load outliers.csv. "

        "STEP 2 — VALIDATE INPUT: "
        "Required columns: route, anomaly_score, ratio_to_baseline, alert_rate, total_alarms. "
        "If missing, raise ValueError. "

        "STEP 3 — RISK SCORING: "
        "Normalize anomaly_score between 0 and 100: "
        "min_score = df['anomaly_score'].min() "
        "max_score = df['anomaly_score'].max() "
        "df['risk_score'] = 100 * (df['anomaly_score'] - min_score) / (max_score - min_score + 1e-9) "

        "STEP 4 — RISK LEVEL: "
        "Assign risk_level using normalized risk_score only: "
        "if risk_score >= 70 → 'HIGH' "
        "elif risk_score >= 30 → 'MEDIUM' "
        "else → 'LOW' "
        "Do not assign HIGH directly from ratio_to_baseline, because ratio_to_baseline may be inflated in sparse route data. "

        "STEP 5 — EXPLANATION: "
        "Create risk_reason column based on BOTH risk_level and signals: "

        "if risk_level == 'HIGH': "
        "    if ratio_to_baseline is high → 'Severely elevated alert rate compared to baseline' "
        "    elif alert_rate is high → 'Very high alert rate on this route' "
        "    else → 'Critical anomaly driven by combined signals' "

        "elif risk_level == 'MEDIUM': "
        "    if ratio_to_baseline is high → 'Elevated alert rate compared to baseline' "
        "    elif alert_rate is high → 'Moderately high alert rate' "
        "    else → 'Moderate anomaly across multiple indicators' "

        "else: "
        "    'Lower priority anomaly; monitor for potential escalation' "

        "STEP 6 — OUTPUT: "
        "Sort by risk_score descending. "
        "Write risk_report.csv. "

        "STEP 7 — SUMMARY: "
        "Print number of HIGH, MEDIUM, LOW risks and top 10 rows. "
    )

## Run: Risk Profiling Agent

This cell executes the Risk Profiling Agent through the supervisor.

- The supervisor:
  - generates and runs the LLM-produced code
  - validates the resulting `risk_report.csv`
  - retries if needed

- If successful, it:
  - loads the risk report
  - prints dataset shape
  - shows the top routes ranked by risk

This step produces the final risk assessment for all detected anomalous routes.

In [787]:
risk_output_csv = os.path.join(OUTPUT_DIR, "risk_report.csv")

result_risk = run_agent_with_supervisor(
    task_name="risk_profiling",
    prompt=_build_risk_prompt(),
    validator_fn=validate_risk_output,
    output_path=risk_output_csv,
    max_retries=3,
)

if result_risk["succeeded"]:
    df_risk = pd.read_csv(risk_output_csv)
    print(f"\nrisk_report.csv: shape={df_risk.shape}")
    print(df_risk.head(10))

  AGENT: risk_profiling

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading outliers.csv...
  │ Data loaded.
  │ Calculating risk scores...
  │ Sorting and writing risk_report.csv...
  │ Report written to /Users/stefanolosurdo/Desktop/STUDIO/LUISS/MACHINELEARNING/FBI Agents/FBI-Agents-817541/output/risk_report.csv
  │ 
  │ Risk Summary:
  │ HIGH risk routes: 2
  │ MEDIUM risk routes: 0
  │ LOW risk routes: 16
  │ 
  │ Top 10 routes by risk score:
  │       route  risk_score risk_level  \
  │ 10  lgw→mxp  100.000000       HIGH   
  │ 8   lcy→flr   97.372154       HIGH   
  │ 12  lhr→lin   10.596697        LOW   
  │ 17  tun→blq    8.020435        LOW   
  │ 3   doh→mxp    1.224146        LOW   
  │ 14  stn→bgy    0.904717        LOW   
  │ 4   eze→fco    0.746987        LOW   
  │ 0   add→fco    0.683498        LOW   
  │ 15  stn→cia    0.668618        LOW   
  │ 1   auh→fco    0.628937        LOW   
  │ 
  │                                           risk_reason  
  │ 10        

## Report Agent

This agent generates the final narrative Transit Anomaly Report.

- It loads `risk_report.csv`
- Safely analyzes only the columns available in the dataset
- Derives departure and arrival airports from the `route` field when possible
- Summarizes:
  - total flagged routes
  - HIGH, MEDIUM, and LOW risk distribution
  - top risky routes
  - main anomaly drivers
- Produces a discursive markdown report written in business/security language
- Saves the final output as `transit_anomaly_report.md`

This step transforms structured risk results into a human-readable operational report.

In [788]:
def _build_report_prompt():
    return (
        "You are a senior intelligence analyst writing a Transit Anomaly Report. "
        "Your task is to produce a detailed narrative report from risk_report.csv. "

        f"Input: '{OUTPUT_DIR}/risk_report.csv'. "
        f"Output: '{OUTPUT_DIR}/transit_anomaly_report.md'. "

        "Generate Python code using pandas and os only. "

        "STEP 1 — LOAD DATA: "
        "Load risk_report.csv. "

        "STEP 2 — SAFE ANALYSIS: "
        "Use only columns that exist in the file. "
        "Do NOT assume columns named departure_airport or arrival_airport exist. "
        "If route exists, derive departure and arrival by splitting route on '→'. "
        "If splitting fails, continue without crashing. "

        "Compute: total flagged routes, HIGH count, MEDIUM count, LOW count, "
        "top routes by risk_score, recurring departure patterns from route prefix, "
        "and average values for alert_rate, total_alarms, anomaly_score, and risk_score when present. "

        "STEP 3 — WRITE MARKDOWN REPORT: "
        "The markdown report MUST include exactly these section headers: "
        "# Transit Anomaly Report "
        "## Executive Summary "
        "## Risk Distribution "
        "## Top High-Risk Routes "
        "## Main Drivers "
        "## Recommended Next Actions "

        "Write in a narrative, discursive style. "
        "Avoid making it only tables and bullet points. "
        "Use full paragraphs and explain the meaning of the results. "

        "In Executive Summary, explain the overall anomaly landscape and how many routes were flagged. "

        "In Risk Distribution, describe how many routes are HIGH, MEDIUM, and LOW risk, "
        "and explain what these levels mean operationally. "

        "In Top High-Risk Routes, discuss the top 5 to 10 routes by risk_score. "
        "Mention concrete route names, risk_score, alert_rate, total_alarms, and risk_reason. "
        "When displaying alert_rate, keep it as a plain numeric value with two decimals, not a percentage. "

        "In Main Drivers, explain how a route is judged risky. "
        "Explain in plain language the role of anomaly_score, alert_rate, total_alarms, "
        "z_score, and ratio_to_baseline if these columns exist. "
        "Clarify that a risky route is not automatically a confirmed threat, but a route whose observed behavior "
        "deviates from expected operational patterns. "

        "In Recommended Next Actions, write a paragraph recommending monitoring, operational review, "
        "focused inspection of HIGH risk routes, and periodic model recalibration. "

        "The report should be at least 800 words if enough data is available. "
        "Do not invent columns. Do not crash if some optional columns are missing. "

        "STEP 4 — SAVE: "
        "Write the report to transit_anomaly_report.md. "
        "Print the output path and a short preview. "
    )

## Run: Report Agent

This cell executes the Report Agent through the supervisor.

- The supervisor:
  - generates and executes the report-generation code
  - validates the final markdown report
  - retries if necessary

- If successful, it:
  - loads the generated `transit_anomaly_report.md`
  - prints a preview of the report content

This step produces the final narrative report summarizing the anomaly and risk analysis.

In [ ]:
report_output_md = os.path.join(OUTPUT_DIR, "transit_anomaly_report.md")

result_report = run_agent_with_supervisor(
    task_name="report",
    prompt=_build_report_prompt(),
    validator_fn=validate_report_output,
    output_path=report_output_md,
    max_retries=3,
)

if result_report["succeeded"]:
    with open(report_output_md, "r", encoding="utf-8") as f:
        report_text = f.read()

    print("\ntransit_anomaly_report.md")
    print(report_text[:2000])

  AGENT: report

  [attempt 1/4] temperature=0.0
  ── stdout ──
  │ Loading risk_report.csv...
  │ Loaded 18 records with columns: ['route', 'total_alarms', 'n_records', 'alert_rate', 'rolling_mean_alarms', 'rolling_std_alarms', 'z_score', 'ratio_to_baseline', 'anomaly_score', 'is_outlier', 'risk_score', 'risk_level', 'risk_reason']
  │ Available columns: ['route', 'total_alarms', 'n_records', 'alert_rate', 'rolling_mean_alarms', 'rolling_std_alarms', 'z_score', 'ratio_to_baseline', 'anomaly_score', 'is_outlier', 'risk_score', 'risk_level', 'risk_reason']
  │ Analysis completed.
  │ Generating report...
  │ Writing report to transit_anomaly_report.md...
  │ # Transit Anomaly Report
  │ 
  │ ## Executive Summary
  │ 
  │ The Transit Anomaly Report provides a comprehensive assessment of flagged transit routes based on behavioral anomalies and risk indicators. This analysis covers a total of 18 flagged routes, each representing a potential deviation from established operational baselines.

In [790]:
baseline_path = os.path.join(OUTPUT_DIR, "baseline_data.csv")
outliers_path = os.path.join(OUTPUT_DIR, "outliers.csv")
risk_path = os.path.join(OUTPUT_DIR, "risk_report.csv")
report_path = os.path.join(OUTPUT_DIR, "transit_anomaly_report.md")

def load_outputs():
    baseline = pd.read_csv(baseline_path)
    outliers = pd.read_csv(outliers_path)
    risk = pd.read_csv(risk_path)

    with open(report_path, "r", encoding="utf-8") as f:
        report = f.read()

    return (
        baseline.head(50),
        outliers.head(50),
        risk.head(50),
        report
    )

with gr.Blocks(title="Transit Anomaly Detection") as demo:
    gr.Markdown("# Transit Anomaly Detection — Multi-Agent Pipeline")

    refresh = gr.Button("Load / Refresh Results")

    with gr.Tab("Baseline"):
        baseline_table = gr.Dataframe(label="baseline_data.csv", interactive=False)

    with gr.Tab("Outliers"):
        outliers_table = gr.Dataframe(label="outliers.csv", interactive=False)

    with gr.Tab("Risk Report"):
        risk_table = gr.Dataframe(label="risk_report.csv", interactive=False)

    with gr.Tab("Final Report"):
        report_md = gr.Markdown()

    refresh.click(
        fn=load_outputs,
        inputs=[],
        outputs=[baseline_table, outliers_table, risk_table, report_md]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
